In [1]:
# Cell 1 - Imports
import os
import time
import joblib
import numpy as np
import pandas as pd
import fastf1

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)


In [2]:
# Cell 2 — Paths and cache config
BASE_DIR    = os.path.dirname(os.getcwd())           # goes up one level to F1/
DATA_PATH   = os.path.join(BASE_DIR, "data",   "f1_dataset_clean.csv")
MODEL_PATH  = os.path.join(BASE_DIR, "models", "model_v9.pkl")  
CACHE_PATH  = os.path.join(BASE_DIR, "cache")

os.makedirs(CACHE_PATH,                     exist_ok=True)
os.makedirs(os.path.dirname(DATA_PATH),     exist_ok=True)
os.makedirs(os.path.dirname(MODEL_PATH),    exist_ok=True)

fastf1.Cache.enable_cache(CACHE_PATH)
print("Base dir:", BASE_DIR)
print("Cache enabled at:", CACHE_PATH)

Base dir: /Users/aakash/Documents/academics/F1
Cache enabled at: /Users/aakash/Documents/academics/F1/cache


In [3]:
# Cell 3 — Track type lookup
TRACK_TYPE = {
    # Street circuits
    "Jeddah":        "street",
    "Baku":          "street",
    "Miami":         "street",
    "Monaco":        "street",
    "Marina Bay":    "street",
    "Las Vegas":     "street",
    "Melbourne":     "street",
    "Miami Gardens": "street",
    "Madrid":        "street",

    # Permanent circuits
    "Sakhir":            "permanent",
    "Barcelona":         "permanent",
    "Montréal":          "permanent",
    "Spielberg":         "permanent",
    "Silverstone":       "permanent",
    "Budapest":          "permanent",
    "Spa-Francorchamps": "permanent",
    "Zandvoort":         "permanent",
    "Monza":             "permanent",
    "Suzuka":            "permanent",
    "Lusail":            "permanent",
    "Austin":            "permanent",
    "Mexico City":       "permanent",
    "São Paulo":         "permanent",
    "Yas Island":        "permanent",
    "Shanghai":          "permanent",
    "Imola":             "permanent",
}

In [6]:
# Cell 4 — fetch_round function
def fetch_round(year, round_num):
    print(f"  Fetching {year} R{round_num}...")

    # --- Race ---
    try:
        session_r = fastf1.get_session(year, round_num, "R")
        session_r.load(laps=False, telemetry=False, weather=False, messages=False)
        race = session_r.results[["FullName", "TeamName", "GridPosition", "Position"]].copy()
        race["Year"]     = year
        race["Round"]    = round_num
        race["Location"] = session_r.event["Location"]
    except Exception as e:
        print(f"  ✗ Race failed: {e}")
        return None

    # --- Qualifying ---
    try:
        session_q = fastf1.get_session(year, round_num, "Q")
        session_q.load(laps=False, telemetry=False, weather=False, messages=False)
        q = session_q.results[["FullName", "Q1", "Q2", "Q3"]].copy()
        q["BestQualiTime"] = q[["Q1", "Q2", "Q3"]].min(axis=1).dt.total_seconds()
        q = q[["FullName", "BestQualiTime"]]
    except Exception as e:
        print(f"  ✗ Quali failed: {e}")
        q = pd.DataFrame(columns=["FullName", "BestQualiTime"])

    # --- Merge ---
    df = race.merge(q, on="FullName", how="left")
    df["Podium"]    = (df["Position"] <= 3).astype(int)
    df["TrackType"] = df["Location"].map(TRACK_TYPE)

    if df["TrackType"].isnull().any():
        unknown = df[df["TrackType"].isnull()]["Location"].unique()
        print(f"  ⚠ Unknown locations — add to TRACK_TYPE: {unknown}")

    print(f"  ✓ {year} R{round_num} — {len(df)} drivers")
    time.sleep(1)
    return df

In [ ]:
# Cell 5 — Fetch all historical data (schedule-aware)
HISTORICAL_YEARS = [2023, 2024, 2025, 2026]  

frames = []

for year in HISTORICAL_YEARS:
    print(f"\n── {year} ──────────────────────")
    
    # Ask FastF1 how many rounds this year actually has
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rounds = schedule["RoundNumber"].tolist()
    print(f"  {len(rounds)} rounds found")
    
    for round_num in rounds:
        df = fetch_round(year, round_num)
        if df is not None:
            frames.append(df)

df_raw = pd.concat(frames, ignore_index=True)

print(f"\nDone.")
print(f"Total rows : {len(df_raw)}")
print(f"Years      : {sorted(df_raw['Year'].unique())}")
print(f"Rounds     : {df_raw.groupby('Year')['Round'].nunique().to_dict()}")

In [ ]:
# Cell 6 — Inspect raw data
print("Shape:", df_raw.shape)
print("\nData types:")
print(df_raw.dtypes)
print("\nMissing values:")
print(df_raw.isnull().sum())
print("\nSample rows:")
df_raw.head(5)

Shape: (1684, 10)

Data types:
FullName          object
TeamName          object
GridPosition     float64
Position         float64
Year               int64
Round              int64
Location          object
BestQualiTime    float64
Podium             int64
TrackType         object
dtype: object

Missing values:
FullName          0
TeamName          0
GridPosition     23
Position         23
Year              0
Round             0
Location          0
BestQualiTime    40
Podium            0
TrackType        22
dtype: int64

Sample rows:


,FullName,TeamName,GridPosition,Position,Year,Round,Location,BestQualiTime,Podium,TrackType
0,Max Verstappen,Red Bull Racing,1.0,1.0,2023,1,Sakhir,89.708,1,permanent
1,Sergio Perez,Red Bull Racing,2.0,2.0,2023,1,Sakhir,89.846,1,permanent
2,Fernando Alonso,Aston Martin,5.0,3.0,2023,1,Sakhir,90.336,1,permanent
3,Carlos Sainz,Ferrari,4.0,4.0,2023,1,Sakhir,90.154,0,permanent
4,Lewis Hamilton,Mercedes,7.0,5.0,2023,1,Sakhir,90.384,0,permanent


In [20]:
# Cell 7 — Clean the data
def clean(df):
    df = df.copy()

    # Drop rows where we don't know the race result
    df = df.dropna(subset=["Position"])

    # Cast to int now that NaNs are gone
    df["Position"] = df["Position"].astype(int)
    df["GridPosition"] = df["GridPosition"].fillna(20).astype(int)

    # 1. Try to impute with the worst time in that specific round
    worst_per_round = df.groupby(["Year", "Round"])["BestQualiTime"].transform("max")
    df["BestQualiTime"] = df["BestQualiTime"].fillna(worst_per_round + 5.0)

    # 2. THE FIX: If the *entire* round was missing, fallback to the global worst time
    global_worst = df["BestQualiTime"].max()
    df["BestQualiTime"] = df["BestQualiTime"].fillna(global_worst + 5.0)

    # Sort chronologically
    df = df.sort_values(["Year", "Round", "Position"]).reset_index(drop=True)

    return df


df_clean = clean(df_raw)

print("Rows before:", len(df_raw))
print("Rows after :", len(df_clean))
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

Rows before: 1684
Rows after : 1661

Missing values after cleaning:
FullName          0
TeamName          0
GridPosition      0
Position          0
Year              0
Round             0
Location          0
BestQualiTime     0
Podium            0
TrackType        22
dtype: int64


In [21]:
# Cell 8 - Feature engineering (v9 enhanced, leakage-safe)
def other_driver_mean(df, by, column):
    counts = df.groupby(by)[column].transform("count")
    totals = df.groupby(by)[column].transform("sum")
    peer_mean = (totals - df[column]) / (counts - 1)
    return peer_mean.where(counts > 1, df[column])


def engineer_features(df):
    df = df.copy()

    # Qualifying features available before the race.
    round_key = ["Year", "Round"]
    team_round_key = ["Year", "Round", "TeamName"]

    df["QualiGapToPole"] = df.groupby(round_key)["BestQualiTime"].transform(
        lambda x: x - x.min()
    )
    df["QualiGapNormalized"] = df.groupby(round_key)["BestQualiTime"].transform(
        lambda x: (x - x.min()) / x.min() * 100
    )
    df["GridPositionSquared"] = df["GridPosition"] ** 2
    df["MidfieldFlag"] = ((df["GridPosition"] >= 8) & (df["GridPosition"] <= 15)).astype(int)

    # Historical driver race craft. Current race results are shifted out.
    df["PositionGain"] = df["GridPosition"] - df["Position"]
    df = df.sort_values(["FullName", "Year", "Round"])
    df["FinishStdLast5"] = (
        df.groupby(["FullName", "Year"])["Position"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).std())
        .fillna(5.0)
    )
    df["AvgFinishLast3"] = (
        df.groupby(["FullName", "Year"])["Position"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        .fillna(10.0)
    )
    df["PodiumRateLast5"] = (
        df.groupby(["FullName", "Year"])["Podium"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        .fillna(0.15)
    )


    # Constructor form must be calculated once per team-race, shifted by race,
    # then merged back so both teammates receive the same pre-race value.
    team_round = (
        df.groupby(["TeamName", "Year", "Round"], as_index=False)
        .agg(
            TeamPodiumCurrent=("Podium", "mean"),
            TeamAvgFinishCurrent=("Position", "mean"),
        )
        .sort_values(["TeamName", "Year", "Round"])
    )
    team_round["ConstructorPodiumRate"] = (
        team_round.groupby(["TeamName", "Year"])["TeamPodiumCurrent"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean())
        .fillna(0.1)
    )


    df = df.merge(
        team_round[["TeamName", "Year", "Round", "ConstructorPodiumRate"]],
        on=["TeamName", "Year", "Round"],
        how="left",
    )

    df = df.sort_values(["Year", "Round", "GridPosition"]).reset_index(drop=True)

    teammate_position = other_driver_mean(df, team_round_key, "Position")
    df["BeatTeammate"] = (df["Position"] < teammate_position).astype(int)
    df["BeatTeammateRate"] = (
        df.groupby(["FullName", "Year"])["BeatTeammate"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean())
        .fillna(0.5)
    )


    return df

df_features = engineer_features(df_clean)

new_cols = [
    "QualiGapToPole", "QualiGapNormalized", "GridPositionSquared", "MidfieldFlag", 
    "FinishStdLast5", "AvgFinishLast3", "PodiumRateLast5", "ConstructorPodiumRate", 
    "GridPosition", "GridPositionSquared", "BeatTeammateRate", "TrackType"
]
print("Missing values in engineered features:")
print(df_features[new_cols].isnull().sum())
print(f"\nTotal engineered features: {len(new_cols)}")


Missing values in engineered features:
QualiGapToPole            0
QualiGapNormalized        0
GridPositionSquared       0
MidfieldFlag              0
FinishStdLast5            0
AvgFinishLast3            0
PodiumRateLast5           0
ConstructorPodiumRate     0
GridPosition              0
GridPositionSquared       0
BeatTeammateRate          0
TrackType                22
dtype: int64

Total engineered features: 12


In [22]:
# Cell 9 — Save featured dataset
df_features.to_csv(DATA_PATH, index=False)
print(f"Saved {len(df_features)} rows to {DATA_PATH}")
print(f"Columns: {df_features.columns.tolist()}")

Saved 1661 rows to /Users/aakash/Documents/academics/F1/data/f1_dataset_clean.csv
Columns: ['FullName', 'TeamName', 'GridPosition', 'Position', 'Year', 'Round', 'Location', 'BestQualiTime', 'Podium', 'TrackType', 'QualiGapToPole', 'QualiGapNormalized', 'GridPositionSquared', 'MidfieldFlag', 'PositionGain', 'FinishStdLast5', 'AvgFinishLast3', 'PodiumRateLast5', 'ConstructorPodiumRate', 'BeatTeammate', 'BeatTeammateRate']


In [23]:
# Cell 10 - Define feature columns and target

FEATURE_COLS = [
    # Qualifying/grid signal
    "GridPosition",
    "GridPositionSquared",
    "QualiGapNormalized",
    "QualiGapToPole",
    "MidfieldFlag",

    # Historical race craft
    "FinishStdLast5",

    # Driver form
    "AvgFinishLast3",
    "PodiumRateLast5",
    "BeatTeammateRate",

    # Leakage-safe constructor form
    "ConstructorPodiumRate",

    # Track type
    "TrackType_street",
    "TrackType_permanent",
]

TARGET = "Podium"

feature_groups = {
    "Qualifying/grid": 5,
    "Historical race craft": 1,
    "Driver form": 3,
    "Constructor form": 1,
    "Track": 2,
}

print(f"Default features: {len(FEATURE_COLS)}")
for label, count in feature_groups.items():
    print(f"  {label:22s}: {count:2d} ({count / len(FEATURE_COLS) * 100:.0f}%)")



Default features: 12
  Qualifying/grid       :  5 (42%)
  Historical race craft :  1 (8%)
  Driver form           :  3 (25%)
  Constructor form      :  1 (8%)
  Track                 :  2 (17%)


In [24]:
# Cell 11 — One-hot encode and split
df_model = pd.get_dummies(df_features, columns=["TrackType"], dtype=int)

# Ensure both columns exist even if one track type is missing from a subset
for col in ["TrackType_street", "TrackType_permanent"]:
    if col not in df_model.columns:
        df_model[col] = 0

# Split: train on 2023+2024+2025, test on 2026
train_df = df_model[df_model["Year"] < 2026]
test_df  = df_model[df_model["Year"] == 2026]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f"Train: {len(X_train)} rows ({train_df['Year'].nunique()} years)")
print(f"Test : {len(X_test)} rows (2026 only)")
print(f"\nPodium rate in train: {y_train.mean():.3f}")
print(f"Podium rate in test : {y_test.mean():.3f}")

Train: 1397 rows (3 years)
Test : 264 rows (2026 only)

Podium rate in train: 0.150
Podium rate in test : 0.136


In [25]:
# Cell 12 - Evaluation helpers and rolling backtest

def make_base_model():
    return GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        min_samples_leaf=10,
        subsample=0.9,
        random_state=42,
    )


def top3_hits(eval_df, score_col, largest=True):
    correct, total = 0, 0
    per_round = []

    for (yr, rnd), grp in eval_df.groupby(["Year", "Round"]):
        ranked = grp.nlargest(3, score_col) if largest else grp.nsmallest(3, score_col)
        pred_top3 = set(ranked["FullName"])
        actual_top3 = set(grp[grp[TARGET] == 1]["FullName"])
        hits = len(pred_top3 & actual_top3)
        possible = min(3, len(actual_top3))
        correct += hits
        total += possible
        per_round.append({"Year": yr, "Round": rnd, "Hits": hits, "Possible": possible})

    return correct, total, correct / total if total else np.nan, pd.DataFrame(per_round)


def grid_top3_baseline(eval_df):
    return top3_hits(eval_df, "GridPosition", largest=False)


def evaluate_predictions(eval_df, y_true, proba, label):
    eval_df = eval_df.copy()
    eval_df["proba"] = proba
    pred = (proba >= 0.5).astype(int)

    print(f"-- {label} --")
    print(classification_report(y_true, pred, zero_division=0))

    metrics = {}
    if y_true.nunique() > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, proba)
        metrics["average_precision"] = average_precision_score(y_true, proba)
        metrics["brier"] = brier_score_loss(y_true, proba)
        print(f"ROC AUC:           {metrics['roc_auc']:.4f}")
        print(f"Average Precision: {metrics['average_precision']:.4f}")
        print(f"Brier Score:       {metrics['brier']:.4f}")

    model_correct, model_total, model_rate, model_rounds = top3_hits(eval_df, "proba", largest=True)
    grid_correct, grid_total, grid_rate, grid_rounds = grid_top3_baseline(eval_df)

    print(f"Model Top-3:        {model_correct}/{model_total} ({model_rate:.1%})")
    print(f"Grid baseline Top-3:{grid_correct}/{grid_total} ({grid_rate:.1%})")

    metrics.update({
        "model_top3_correct": model_correct,
        "model_top3_total": model_total,
        "model_top3_rate": model_rate,
        "grid_top3_correct": grid_correct,
        "grid_top3_total": grid_total,
        "grid_top3_rate": grid_rate,
    })
    return metrics, model_rounds.merge(
        grid_rounds,
        on=["Year", "Round"],
        suffixes=("Model", "Grid"),
    )


def rolling_backtest(df_model, feature_cols):
    scenarios = [
        {"train_years": [2023], "test_year": 2024},
        {"train_years": [2023, 2024], "test_year": 2025},
        {"train_years": [2023, 2024, 2025], "test_year": 2026},
    ]
    rows = []

    for scenario in scenarios:
        scenario_train = df_model[df_model["Year"].isin(scenario["train_years"])]
        scenario_test = df_model[df_model["Year"] == scenario["test_year"]]
        if scenario_train.empty or scenario_test.empty or scenario_test[TARGET].nunique() < 2:
            continue

        weights = compute_sample_weights(scenario_train, max_year=max(scenario["train_years"]))
        backtest_model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
        backtest_model.fit(
            scenario_train[feature_cols],
            scenario_train[TARGET],
            sample_weight=weights,
        )

        scored = scenario_test.copy()
        scored["proba"] = backtest_model.predict_proba(scenario_test[feature_cols])[:, 1]
        model_correct, model_total, model_rate, _ = top3_hits(scored, "proba", largest=True)
        grid_correct, grid_total, grid_rate, _ = grid_top3_baseline(scored)

        rows.append({
            "TrainYears": "+".join(str(y) for y in scenario["train_years"]),
            "TestYear": scenario["test_year"],
            "ROC_AUC": roc_auc_score(scenario_test[TARGET], scored["proba"]),
            "AveragePrecision": average_precision_score(scenario_test[TARGET], scored["proba"]),
            "Brier": brier_score_loss(scenario_test[TARGET], scored["proba"]),
            "ModelTop3": f"{model_correct}/{model_total}",
            "ModelTop3Rate": model_rate,
            "GridTop3": f"{grid_correct}/{grid_total}",
            "GridTop3Rate": grid_rate,
        })

    return pd.DataFrame(rows)


In [ ]:
# Cell 13 - Era/sample weights
DECAY_FACTOR = 0.6


def compute_sample_weights(df, max_year):
    gap = max_year - df["Year"]
    weights = DECAY_FACTOR ** gap
    return weights.values


train_weights = compute_sample_weights(train_df, max_year=train_df["Year"].max())

all_weights = compute_sample_weights(df_model, max_year=2026)
for year in sorted(df_model["Year"].unique()):
    mask = df_model["Year"] == year
    print(f"{year}: display weight = {all_weights[mask][0]:.4f}  ({mask.sum()} rows)")

for year in sorted(train_df["Year"].unique()):
    mask = train_df["Year"] == year
    print(f"train {year}: weight = {train_weights[mask][0]:.4f}  ({mask.sum()} rows)")


2023: display weight = 0.2160  (439 rows)
2024: display weight = 0.3600  (479 rows)
2025: display weight = 0.6000  (479 rows)
2026: display weight = 1.0000  (264 rows)
train 2023: weight = 0.3600  (439 rows)
train 2024: weight = 0.6000  (479 rows)
train 2025: weight = 1.0000  (479 rows)


In [ ]:
# Cell 14 - Train base model with era weights
base_model = make_base_model()
base_model.fit(X_train, y_train, sample_weight=train_weights)
print("Base model trained")


Base model trained


In [ ]:
# Cell 15 - Probability calibration
model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
model.fit(X_train, y_train, sample_weight=train_weights)
print("Calibrated model ready")


Calibrated model ready


In [ ]:
# Cell 16 - Evaluation
proba = model.predict_proba(X_test)[:, 1]
metrics_2026, per_round_2026 = evaluate_predictions(test_df, y_test, proba, "2026 holdout")

print("\nPer-round Top-3 hits:")
print(per_round_2026)

print("\nRolling backtest:")
rolling_results = rolling_backtest(df_model, FEATURE_COLS)
print(rolling_results)


-- 2026 holdout --
              precision    recall  f1-score   support

           0       0.91      0.94      0.93       228
           1       0.55      0.44      0.49        36

    accuracy                           0.88       264
   macro avg       0.73      0.69      0.71       264
weighted avg       0.87      0.88      0.87       264

ROC AUC:           0.9221
Average Precision: 0.5883
Brier Score:       0.0772
Model Top-3:        21/36 (58.3%)
Grid baseline Top-3:22/36 (61.1%)

Per-round Top-3 hits:
    Year  Round  HitsModel  PossibleModel  HitsGrid  PossibleGrid
0   2026      1          3              3         2             3
1   2026      2          2              3         3             3
2   2026      3          2              3         2             3
3   2026      4          1              3         1             3
4   2026      5          1              3         1             3
5   2026      6          1              3         2             3
6   2026      7        

In [ ]:
# Cell 17 — Feature importance
print("-- Feature Importance --")
for feat, imp in sorted(
    zip(FEATURE_COLS, base_model.feature_importances_), key=lambda x: -x[1]
):
    print(f"  {feat:25s} {imp:.4f}")


-- Feature Importance --
  GridPosition              0.3794
  GridPositionSquared       0.1764
  QualiGapNormalized        0.0880
  FinishStdLast5            0.0826
  AvgFinishLast3            0.0816
  QualiGapToPole            0.0802
  ConstructorPodiumRate     0.0507
  PodiumRateLast5           0.0309
  BeatTeammateRate          0.0153
  TrackType_permanent       0.0067
  MidfieldFlag              0.0045
  TrackType_street          0.0038


In [ ]:
# Cell 18 — Save model
joblib.dump(model, MODEL_PATH)
print(f"✓ Model saved → {MODEL_PATH}")
